# 13.4 Text-to-Speech synthesis

TTS turns text into sound by predicting what to say, how long to hold it, and how the acoustic contour should move.

Duration expansion maps short token sequences into many acoustic frames, mel targets avoid raw waveform phase, and pitch/energy contours carry prosody.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(1301)


def sine_wave(freq, duration, fs, amplitude=1.0, phase=0.0):
    n = np.arange(int(round(duration * fs)))
    x = amplitude * np.sin(2.0 * np.pi * freq * n / fs + phase)
    return x


def chirp_wave(start_freq, end_freq, duration, fs, amplitude=1.0):
    n = np.arange(int(round(duration * fs)))
    t = n / fs
    slope = (end_freq - start_freq) / duration
    phase = 2.0 * np.pi * (start_freq * t + 0.5 * slope * t * t)
    x = amplitude * np.sin(phase)
    return x


def rms(x):
    return float(np.sqrt(np.mean(np.square(x))))


def add_noise_for_snr(x, snr_db, seed):
    local_rng = np.random.default_rng(seed)
    noise = local_rng.normal(size=len(x))
    target_noise_rms = rms(x) / (10.0 ** (snr_db / 20.0))
    noise = noise / (rms(noise) + 1e-12)
    y = x + target_noise_rms * noise
    return y


def frame_signal(x, n_fft, hop):
    if len(x) < n_fft:
        pad_width = n_fft - len(x)
        x = np.pad(x, (0, pad_width))
    starts = np.arange(0, len(x) - n_fft + 1, hop)
    frames = np.stack([x[start:start + n_fft] for start in starts])
    return frames


def stft_mag(x, n_fft, hop):
    frames = frame_signal(x, n_fft, hop)
    window = np.hanning(n_fft)
    spectrum = np.fft.rfft(frames * window[None, :], axis=1)
    magnitude = np.abs(spectrum)
    return magnitude


def hz_to_mel(freq):
    return 2595.0 * np.log10(1.0 + freq / 700.0)


def mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)


def mel_filterbank(fs, n_fft, n_mels):
    freqs = np.linspace(0.0, fs / 2.0, n_fft // 2 + 1)
    mel_edges = np.linspace(hz_to_mel(0.0), hz_to_mel(fs / 2.0), n_mels + 2)
    hz_edges = mel_to_hz(mel_edges)
    bank = np.zeros((n_mels, len(freqs)))
    for band in range(n_mels):
        left = hz_edges[band]
        center = hz_edges[band + 1]
        right = hz_edges[band + 2]
        rising = (freqs - left) / max(center - left, 1e-12)
        falling = (right - freqs) / max(right - center, 1e-12)
        bank[band] = np.maximum(0.0, np.minimum(rising, falling))
    return bank


def dct_matrix(n):
    rows = []
    for k in range(n):
        row = []
        for b in range(n):
            value = math.cos(math.pi * k * (2 * b + 1) / (2 * n))
            row.append(value)
        rows.append(row)
    return np.array(rows)


def build_audio_ladder(fs=8000):
    d1 = sine_wave(440.0, 0.45, fs)
    d2 = 0.65 * sine_wave(440.0, 0.50, fs) + 0.35 * sine_wave(660.0, 0.50, fs)
    d3_clean = 0.60 * sine_wave(440.0, 0.55, fs) + 0.35 * sine_wave(880.0, 0.55, fs)
    d3 = add_noise_for_snr(d3_clean, 12.0, 1303)
    d4 = np.concatenate([
        sine_wave(330.0, 0.20, fs),
        chirp_wave(360.0, 900.0, 0.25, fs),
        0.80 * sine_wave(550.0, 0.20, fs),
        0.65 * sine_wave(720.0, 0.15, fs),
    ])
    d5_clean = np.concatenate([
        0.70 * sine_wave(300.0, 0.25, fs),
        chirp_wave(320.0, 980.0, 0.35, fs),
        0.55 * sine_wave(440.0, 0.25, fs) + 0.25 * sine_wave(880.0, 0.25, fs),
        0.70 * sine_wave(660.0, 0.30, fs),
        chirp_wave(900.0, 420.0, 0.25, fs),
    ])
    d5 = add_noise_for_snr(d5_clean, 5.0, 1305)
    ladder = [
        {"name": "D1 single synthetic sine", "x": d1, "fs": fs, "targets": [440.0], "complexity": 1},
        {"name": "D2 two-tone command", "x": d2, "fs": fs, "targets": [440.0, 660.0], "complexity": 2},
        {"name": "D3 noisy two-tone", "x": d3, "fs": fs, "targets": [440.0, 880.0], "complexity": 3},
        {"name": "D4 chirp multi-segment", "x": d4, "fs": fs, "targets": [330.0, 550.0, 720.0], "complexity": 4},
        {"name": "D5 longer noisier phrase", "x": d5, "fs": fs, "targets": [300.0, 440.0, 660.0], "complexity": 5},
    ]
    return ladder


def dominant_frequencies(x, fs, n_fft=512, hop=128, count=3):
    mag = stft_mag(x, n_fft, hop)
    mean_mag = np.mean(mag, axis=0)
    mean_mag[0] = 0.0
    order = np.argsort(mean_mag)[::-1]
    freqs = order[:count] * fs / n_fft
    return freqs


def nearest_error(predicted, targets):
    predicted = np.array(predicted)
    errors = []
    for target in targets:
        error = float(np.min(np.abs(predicted - target)))
        errors.append(error)
    return float(np.mean(errors))


## The concept, built once (D1): duration expansion

A duration-based TTS model expands token states into $T=\sum_i d_i$ frames before predicting acoustic features.

In [ ]:

tokens = ["h", "e", "l", "o"]
durations = np.array([2, 1, 2, 1])
starts = np.cumsum(np.r_[0, durations[:-1]])
expanded = []
for token, duration in zip(tokens, durations):
    expanded.extend([token] * int(duration))

assert int(np.sum(durations)) == 6
assert starts.tolist() == [0, 2, 3, 5]
assert expanded == ["h", "h", "e", "l", "l", "o"]

print("total frames", int(np.sum(durations)))
print("token starts", starts.tolist())
print("expanded", expanded)


The lesson loss combines mel reconstruction and pitch:

$$\hat M_{t,b}=g(\operatorname{expand}(h_i,d_i)), \qquad \mathcal{L}=\|\hat M-M\|_1+\lambda_f\|\hat f_0-f_0\|_1.$$

The function below creates a small mel contour from token identity, duration, and pitch.

In [ ]:

def synthesize_mel(tokens, durations, pitch, n_mels=24):
    frames = []
    pitch_frames = []
    for idx, token in enumerate(tokens):
        duration = int(durations[idx])
        center = 3 + (sum(ord(ch) for ch in token) % (n_mels - 6))
        mel_axis = np.arange(n_mels)
        base = np.exp(-0.5 * ((mel_axis - center) / 2.2) ** 2)
        for local in range(duration):
            tilt = 1.0 + 0.10 * math.sin((local + 1) / max(duration, 1) * math.pi)
            frames.append(base * tilt)
            pitch_frames.append(float(pitch[idx]))
    mel = np.stack(frames)
    f0 = np.array(pitch_frames)
    return mel, f0

pitch = np.array([180.0, 190.0, 170.0, 160.0])
mel, f0 = synthesize_mel(tokens, durations, pitch)
mean_pitch = round(float(np.mean([180, 190, 210, 200, 170, 160])), 1)
pitch_range = int(np.max([180, 190, 210, 200, 170, 160]) - np.min([180, 190, 210, 200, 170, 160]))
toy_loss = abs(0.7 - 1.0) + 0.01 * 10

assert mel.shape[0] == 6
assert mean_pitch == 185.0
assert pitch_range == 50
assert round(float(toy_loss), 3) == 0.4

print("mel frames", mel.shape)
print("mean pitch", mean_pitch)
print("pitch range", pitch_range)
print("toy loss", round(float(toy_loss), 3))


## The dataset ladder

The F7 audio ladder becomes paired target mel frames: D1 single vowel, D2 two-token phrase, D3 noisy target, D4 chirp/multi-segment phrase, and D5 longer/noisier phrase with duration mismatch.

In [ ]:

def make_tts_ladder():
    ladder = [
        {"name": "D1 single synthetic vowel", "tokens": ["a"], "durations": np.array([6]), "pitch": np.array([180.0]), "noise": 0.0},
        {"name": "D2 two-tone token sequence", "tokens": ["a", "b"], "durations": np.array([4, 4]), "pitch": np.array([180.0, 205.0]), "noise": 0.0},
        {"name": "D3 noisy mel targets", "tokens": ["a", "b"], "durations": np.array([4, 5]), "pitch": np.array([175.0, 215.0]), "noise": 0.04},
        {"name": "D4 chirp multi-segment phrase", "tokens": ["h", "e", "l", "o"], "durations": np.array([3, 2, 4, 3]), "pitch": np.array([180.0, 195.0, 210.0, 170.0]), "noise": 0.03},
        {"name": "D5 longer noisier duration mismatch", "tokens": ["s", "p", "e", "e", "c", "h"], "durations": np.array([3, 2, 4, 3, 4, 3]), "pitch": np.array([170.0, 185.0, 205.0, 210.0, 190.0, 165.0]), "noise": 0.08},
    ]
    for idx, rung in enumerate(ladder):
        target, target_f0 = synthesize_mel(rung["tokens"], rung["durations"], rung["pitch"])
        if rung["noise"] > 0.0:
            local_rng = np.random.default_rng(1340 + idx)
            target = target + local_rng.normal(scale=rung["noise"], size=target.shape)
        rung["target_mel"] = target
        rung["target_f0"] = target_f0
    return ladder

tts_ladder = make_tts_ladder()

for rung in tts_ladder:
    print(rung["name"])
    print("  tokens", rung["tokens"])
    print("  durations", rung["durations"].tolist())
    print("  target shape", rung["target_mel"].shape)
    print("  first mel", np.round(rung["target_mel"][0, :5], 3).tolist())


## Run the SAME method across D1-D5

The same duration expansion and mel synthesis method is applied to every rung, and the metric is mel reconstruction MAE.

In [ ]:

def mel_mae(pred, target):
    length = min(pred.shape[0], target.shape[0])
    return float(np.mean(np.abs(pred[:length] - target[:length])))

def predict_tts(rung, duration_scale=1.0, pitch_scale=1.0):
    pred_durations = np.maximum(1, np.rint(rung["durations"] * duration_scale).astype(int))
    pred_pitch = rung["pitch"] * pitch_scale
    pred_mel, pred_f0 = synthesize_mel(rung["tokens"], pred_durations, pred_pitch)
    return pred_mel, pred_f0, pred_durations

tts_rows = []

for rung in tts_ladder:
    pred_mel, pred_f0, pred_durations = predict_tts(rung)
    mae = mel_mae(pred_mel, rung["target_mel"])
    tts_rows.append({"rung": rung, "pred_mel": pred_mel, "pred_f0": pred_f0, "pred_durations": pred_durations, "metric": mae})

print("rung | mel reconstruction MAE | predicted frames")
for row in tts_rows:
    print(row["rung"]["name"], "|", round(row["metric"], 4), "|", row["pred_mel"].shape[0])


## Results visualization

The closing figure has two parts: target/predicted mel panels for every rung, then MAE vs complexity.

In [ ]:

fig, axes = plt.subplots(5, 2, figsize=(10, 12))
mae_values = []

for idx, row in enumerate(tts_rows):
    target = row["rung"]["target_mel"]
    pred = row["pred_mel"]
    axes[idx, 0].imshow(target.T, origin="lower", aspect="auto", cmap="viridis")
    axes[idx, 0].set_title(row["rung"]["name"] + " target mel")
    axes[idx, 1].imshow(pred.T, origin="lower", aspect="auto", cmap="viridis")
    axes[idx, 1].set_title("predicted mel")
    mae_values.append(row["metric"])

plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), mae_values, marker="o")
plt.xlabel("complexity rung")
plt.ylabel("mel reconstruction MAE")
plt.grid(True)
plt.show()


## Pitfall on D5: duration alignment drift skips or repeats sounds

If the duration sum is wrong, spectrogram frames attach to the wrong token.  Normalize durations back to the target frame count and include pitch loss so prosody is not flat.

In [ ]:

def normalize_durations(durations, target_frames):
    scaled = durations * (target_frames / np.sum(durations))
    normalized = np.maximum(1, np.floor(scaled).astype(int))
    while np.sum(normalized) < target_frames:
        idx = int(np.argmax(scaled - normalized))
        normalized[idx] += 1
    while np.sum(normalized) > target_frames:
        idx = int(np.argmax(normalized))
        if normalized[idx] > 1:
            normalized[idx] -= 1
        else:
            break
    return normalized

d5 = tts_ladder[-1]
wrong_durations = d5["durations"] + np.array([1, 0, 1, 0, 1, 0])
wrong_mel, wrong_f0 = synthesize_mel(d5["tokens"], wrong_durations, np.full_like(d5["pitch"], np.mean(d5["pitch"])))
target_frames = d5["target_mel"].shape[0]
fixed_durations = normalize_durations(wrong_durations, target_frames)
fixed_mel, fixed_f0 = synthesize_mel(d5["tokens"], fixed_durations, d5["pitch"])
wrong_mae = mel_mae(wrong_mel, d5["target_mel"])
fixed_mae = mel_mae(fixed_mel, d5["target_mel"])
wrong_pitch_loss = float(np.mean(np.abs(wrong_f0[:target_frames] - d5["target_f0"])))
fixed_pitch_loss = float(np.mean(np.abs(fixed_f0 - d5["target_f0"])))

print("wrong duration sum", int(np.sum(wrong_durations)), "target", target_frames)
print("fixed duration sum", int(np.sum(fixed_durations)), "target", target_frames)
print("wrong MAE", round(wrong_mae, 4), "fixed MAE", round(fixed_mae, 4))
print("wrong pitch loss", round(wrong_pitch_loss, 3), "fixed pitch loss", round(fixed_pitch_loss, 3))


## Evaluate it + Practice

- Metric: mel reconstruction MAE across D1-D5
- No-skill baseline: guess the most common/simple output and compare against the ladder metric.
- Cheap sanity check: durations [2, 1, 2, 1] must expand to 6 frames with starts [0, 2, 3, 5]
- Ablation: flatten pitch or perturb durations; MAE/pitch loss should increase on D5
- Failure signals: duration sums disagree with target frames, flat f0 range, or low mel loss with lifeless pitch

Practice prompts:

**Practice.** Double one token duration and inspect which mel frames shift.

**Practice.** Increase lambda_f and recompute a combined mel-plus-pitch loss.

**Practice.** Change the token-to-mel center mapping and listen conceptually for identity changes.